# Hyperparameter Optimization: Grid vs Random vs Bayesian

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/optimisation/hyperparameter_optimization.ipynb)

This notebook accompanies the blog post at [sesen.ai](https://sesen.ai/blog/hyperparameter-optimization-grid-random-bayesian).

We compare three strategies for tuning a Random Forest classifier:
1. **Grid Search** — exhaustive evaluation of a predefined grid
2. **Random Search** — uniform random sampling from parameter ranges
3. **Bayesian Optimization (GP)** — model-based sequential search using `scikit-optimize`

In [ ]:
# Install scikit-optimize for Bayesian optimization
!pip install -q scikit-optimize

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (
    cross_val_score, GridSearchCV, RandomizedSearchCV, StratifiedKFold
)
from sklearn import metrics
from skopt import gp_minimize
from skopt.space import Real, Integer, Categorical
import time

## Setup: The Dataset

We generate a synthetic classification problem with 2,000 samples, 20 features (10 informative), and 4 classes. This mirrors the original Kaggle mobile price dataset but works standalone in Colab.

In [ ]:
X, y = make_classification(
    n_samples=2000, n_features=20, n_informative=10,
    n_classes=4, random_state=42
)

print(f'{X.shape[1]} features, {len(np.unique(y))} classes, {X.shape[0]} samples')

# Shared CV strategy
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## Shared Evaluation Function

All methods will tune the same 4 hyperparameters:

| Hyperparameter | Range | What it controls |
|---|---|---|
| `max_features` | [0.1, 1.0] | Fraction of features per split |
| `n_estimators` | [100, 1000] | Number of trees |
| `min_samples_leaf` | [5, 25] | Minimum samples in a leaf |
| `criterion` | {gini, entropy} | Split quality measure |

In [ ]:
def evaluate_params(params):
    """5-fold stratified CV accuracy for a RandomForest configuration.
    Returns negative accuracy (since gp_minimize minimises)."""
    max_features, n_estimators, min_samples_leaf, criterion = params
    model = RandomForestClassifier(
        max_features=max_features,
        n_estimators=n_estimators,
        min_samples_leaf=min_samples_leaf,
        criterion=criterion,
        n_jobs=-1,
        random_state=42
    )
    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    accuracies = []
    for train_idx, val_idx in kf.split(X, y):
        model.fit(X[train_idx], y[train_idx])
        preds = model.predict(X[val_idx])
        accuracies.append(metrics.accuracy_score(y[val_idx], preds))
    return -np.mean(accuracies)

## Method 1: Grid Search

Evaluate every combination on a predefined grid. With 4×2×3×3 = 72 combinations and 5 folds each, that's 360 model fits.

In [ ]:
param_grid = {
    'n_estimators': [200, 400, 600, 800],
    'criterion': ['gini', 'entropy'],
    'min_samples_leaf': [5, 10, 20],
    'max_features': [0.3, 0.5, 0.8]
}

t0 = time.time()
grid_search = GridSearchCV(
    estimator=RandomForestClassifier(n_jobs=-1, random_state=42),
    param_grid=param_grid,
    scoring='accuracy',
    cv=cv,
    verbose=0,
    n_jobs=-1
)
grid_search.fit(X, y)
grid_time = time.time() - t0

print(f'Grid Search — Best accuracy: {grid_search.best_score_:.4f}')
print(f'Combinations evaluated: {len(grid_search.cv_results_["mean_test_score"])}')
print(f'Time: {grid_time:.1f}s')
print(f'Best params: {grid_search.best_params_}')

## Method 2: Random Search

Sample 15 combinations uniformly at random from the parameter ranges.

In [ ]:
param_distributions = {
    'n_estimators': np.arange(100, 1001),
    'criterion': ['gini', 'entropy'],
    'min_samples_leaf': np.arange(5, 26),
    'max_features': np.linspace(0.1, 1.0, 100)
}

t0 = time.time()
random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(n_jobs=-1, random_state=42),
    param_distributions=param_distributions,
    n_iter=15,
    scoring='accuracy',
    cv=cv,
    verbose=0,
    n_jobs=-1,
    random_state=42
)
random_search.fit(X, y)
random_time = time.time() - t0

print(f'Random Search — Best accuracy: {random_search.best_score_:.4f}')
print(f'Combinations evaluated: 15')
print(f'Time: {random_time:.1f}s')
print(f'Best params: {random_search.best_params_}')

## Method 3: Bayesian Optimization (Gaussian Process)

Build a GP surrogate model and use Expected Improvement to decide where to evaluate next. 15 total evaluations: 10 random starts + 5 GP-guided.

In [ ]:
param_space = [
    Real(0.1, 1.0, prior='uniform', name='max_features'),
    Integer(100, 1000, name='n_estimators'),
    Integer(5, 25, name='min_samples_leaf'),
    Categorical(['gini', 'entropy'], name='criterion')
]

t0 = time.time()
result = gp_minimize(
    evaluate_params,
    dimensions=param_space,
    n_calls=15,
    n_random_starts=10,
    random_state=42,
    verbose=False
)
bayes_time = time.time() - t0

best_params = dict(zip(
    ['max_features', 'n_estimators', 'min_samples_leaf', 'criterion'],
    result.x
))
print(f'Bayesian (GP) — Best accuracy: {-result.fun:.4f}')
print(f'Evaluations: 15 (10 random + 5 guided)')
print(f'Time: {bayes_time:.1f}s')
print(f'Best params: {best_params}')

## Results Comparison

In [ ]:
print(f'{"Method":<25} {"Best Accuracy":<16} {"Evaluations":<14} {"Time (s)":<10}')
print('-' * 65)
print(f'{"Grid Search":<25} {grid_search.best_score_:<16.4f} {len(grid_search.cv_results_["mean_test_score"]):<14} {grid_time:<10.1f}')
print(f'{"Random Search":<25} {random_search.best_score_:<16.4f} {15:<14} {random_time:<10.1f}')
print(f'{"Bayesian (GP)":<25} {-result.fun:<16.4f} {15:<14} {bayes_time:<10.1f}')

## Convergence Chart

Track the best accuracy found over successive evaluations for each method.

In [ ]:
# Extract per-evaluation scores
grid_scores = grid_search.cv_results_['mean_test_score']
grid_best = np.maximum.accumulate(grid_scores)

random_scores = random_search.cv_results_['mean_test_score']
random_best = np.maximum.accumulate(random_scores)

bayes_scores = [-y for y in result.func_vals]
bayes_best = np.maximum.accumulate(bayes_scores)

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(range(1, len(grid_best) + 1), grid_best, '-', color='#e74c3c',
        linewidth=2, label=f'Grid Search ({len(grid_best)} evals)')
ax.scatter(range(1, len(grid_scores) + 1), grid_scores, c='#e74c3c', s=15, alpha=0.3)

ax.plot(range(1, len(random_best) + 1), random_best, '-', color='#27ae60',
        linewidth=2.5, label=f'Random Search ({len(random_best)} evals)')
ax.scatter(range(1, len(random_scores) + 1), random_scores, c='#27ae60', s=40, alpha=0.5)

ax.plot(range(1, len(bayes_best) + 1), bayes_best, '-', color='#2980b9',
        linewidth=2.5, label=f'Bayesian GP ({len(bayes_best)} evals)')
ax.scatter(range(1, 11), bayes_scores[:10], c='#2980b9', s=40, alpha=0.5)
ax.scatter(range(11, len(bayes_scores) + 1), bayes_scores[10:], c='#e67e22',
           s=60, alpha=0.8, marker='*', label='GP-guided evaluations')

ax.axvline(x=10.5, color='#2980b9', linestyle=':', alpha=0.4)
ax.set_xlabel('Evaluation Number', fontsize=11)
ax.set_ylabel('Best CV Accuracy So Far', fontsize=11)
ax.set_title('Convergence: Grid vs Random vs Bayesian', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=9)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

## Sampling Comparison

Visualise how each method samples a 2D hyperparameter space. Grid search evaluates fixed lattice points; random search covers more unique values per dimension.

In [ ]:
np.random.seed(42)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

# Grid Search
gx, gy = np.meshgrid([0.2, 0.5, 0.8], [0.2, 0.5, 0.8])
axes[0].scatter(gx, gy, c='#e74c3c', s=80, edgecolors='white', linewidth=1.5)
for x in [0.2, 0.5, 0.8]:
    axes[0].axvline(x, color='#e74c3c', alpha=0.2, linestyle='--')
    axes[0].axhline(x, color='#e74c3c', alpha=0.2, linestyle='--')
axes[0].set_title('Grid Search', fontweight='bold')
axes[0].set_xlabel('Hyperparameter 1')
axes[0].set_ylabel('Hyperparameter 2')

# Random Search
rx, ry = np.random.uniform(0, 1, 9), np.random.uniform(0, 1, 9)
axes[1].scatter(rx, ry, c='#27ae60', s=80, edgecolors='white', linewidth=1.5)
axes[1].set_title('Random Search', fontweight='bold')
axes[1].set_xlabel('Hyperparameter 1')

# Bayesian
bx = np.array([0.15, 0.85, 0.4, 0.6, 0.3, 0.65, 0.72, 0.68, 0.74])
by = np.array([0.8, 0.3, 0.5, 0.2, 0.9, 0.55, 0.4, 0.7, 0.6])
axes[2].scatter(bx[:5], by[:5], c='#3498db', s=80, edgecolors='white',
                linewidth=1.5, label='Random (initial)')
axes[2].scatter(bx[5:], by[5:], c='#e67e22', s=100, edgecolors='white',
                linewidth=1.5, marker='*', label='GP-guided')
axes[2].set_title('Bayesian Optimization', fontweight='bold')
axes[2].set_xlabel('Hyperparameter 1')
axes[2].legend(fontsize=8)

for ax in axes:
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect('equal')

plt.tight_layout()
plt.show()

## GP Surrogate Visualization (1D)

Watch how the Gaussian Process builds a model of the objective function, with the Expected Improvement acquisition function guiding the next evaluation.

In [ ]:
from matplotlib.gridspec import GridSpec
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from scipy.stats import norm

np.random.seed(42)

# 1D objective
def f_true(x):
    return -(np.sin(3*x) * np.sin(x) + 0.3*np.cos(5*x) - 0.5*(x-2)**2/4 + 1.5)

x_plot = np.linspace(0, 4, 500)
y_true = f_true(x_plot)

# Simple GP
def rbf_kernel(x1, x2, ls=0.5, var=1.0):
    x1, x2 = np.atleast_2d(x1).T, np.atleast_2d(x2).T
    sq = np.sum(x1**2, 1).reshape(-1,1) + np.sum(x2**2, 1) - 2*x1@x2.T
    return var * np.exp(-0.5 * sq / ls**2)

def gp_predict(X_tr, y_tr, X_te):
    K = rbf_kernel(X_tr, X_tr) + 1e-6*np.eye(len(X_tr))
    Ks = rbf_kernel(X_tr, X_te)
    Kss = rbf_kernel(X_te, X_te)
    L = np.linalg.cholesky(K)
    a = np.linalg.solve(L.T, np.linalg.solve(L, y_tr))
    mu = (Ks.T @ a).flatten()
    v = np.linalg.solve(L, Ks)
    var = np.maximum(np.diag(Kss - v.T@v), 1e-10)
    return mu, np.sqrt(var)

def ei(mu, sigma, y_best):
    imp = -(mu - y_best)
    Z = imp / (sigma + 1e-10)
    return imp * norm.cdf(Z) + sigma * norm.pdf(Z)

# Sequential observations
obs_x, obs_y = list(np.array([0.5, 3.2])), list(f_true(np.array([0.5, 3.2])))
for _ in range(10):
    mu, sig = gp_predict(np.array(obs_x), np.array(obs_y), x_plot)
    e = ei(mu, sig, min(obs_y))
    nx = x_plot[np.argmax(e)]
    obs_x.append(nx); obs_y.append(f_true(nx))

In [ ]:
fig = plt.figure(figsize=(10, 6))
gs = GridSpec(2, 1, height_ratios=[3, 1], hspace=0.08)
ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1])

def update(frame):
    n = frame + 2
    Xo, yo = np.array(obs_x[:n]), np.array(obs_y[:n])
    mu, sig = gp_predict(Xo, yo, x_plot)
    e = ei(mu, sig, min(yo))

    ax1.clear()
    ax1.plot(x_plot, y_true, 'b-', lw=1.5, alpha=0.3, label='True (unknown)')
    ax1.plot(x_plot, mu, 'k-', lw=2, label='GP mean')
    ax1.fill_between(x_plot, mu-1.96*sig, mu+1.96*sig, alpha=0.15, color='steelblue')
    ax1.scatter(Xo[:-1] if n>2 else Xo, yo[:-1] if n>2 else yo, c='black', s=60, zorder=5)
    if n > 2:
        ax1.scatter([Xo[-1]], [yo[-1]], c='#e67e22', s=100, zorder=6, marker='*')
    best_i = np.argmin(yo)
    ax1.scatter([Xo[best_i]], [yo[best_i]], c='#27ae60', s=100, zorder=6, marker='D')
    ax1.set_xlim(0, 4)
    ax1.set_ylim(min(y_true)-0.5, max(y_true)+0.5)
    ax1.set_title(f'Step {n-2} ({n} observations)', fontweight='bold')
    ax1.set_xticklabels([])
    ax1.grid(True, alpha=0.15)
    ax1.legend(loc='upper right', fontsize=8)

    ax2.clear()
    ax2.fill_between(x_plot, 0, e, alpha=0.4, color='#27ae60')
    ax2.plot(x_plot, e, color='#1a8a4a', lw=1.5)
    if n < 12:
        ni = np.argmax(e)
        ax2.axvline(x_plot[ni], color='#e67e22', ls='--', lw=1.5, alpha=0.8)
    ax2.set_xlim(0, 4)
    ax2.set_xlabel('Hyperparameter value')
    ax2.set_ylabel('EI')
    ax2.grid(True, alpha=0.15)

anim = FuncAnimation(fig, update, frames=11, interval=800)
HTML(anim.to_jshtml())

## Bonus: Optuna (Modern Bayesian HPO)

Optuna uses Tree-structured Parzen Estimators (TPE) instead of GPs, with a more modern API.

In [ ]:
!pip install -q optuna

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def optuna_objective(trial):
    criterion = trial.suggest_categorical('criterion', ['gini', 'entropy'])
    n_estimators = trial.suggest_int('n_estimators', 100, 1000)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 5, 25)
    max_features = trial.suggest_float('max_features', 0.1, 1.0)

    model = RandomForestClassifier(
        n_estimators=n_estimators, criterion=criterion,
        max_features=max_features, min_samples_leaf=min_samples_leaf,
        n_jobs=-1, random_state=42
    )
    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    for train_idx, val_idx in kf.split(X, y):
        model.fit(X[train_idx], y[train_idx])
        scores.append(metrics.accuracy_score(y[val_idx], model.predict(X[val_idx])))
    return np.mean(scores)

study = optuna.create_study(direction='maximize')
study.optimize(optuna_objective, n_trials=15)

print(f'Optuna (TPE) — Best accuracy: {study.best_value:.4f}')
print(f'Best params: {study.best_params}')

## Exercises

1. **More iterations** — Increase `n_calls` to 50 for the Bayesian optimizer. Does the extra budget find meaningfully better configurations?
2. **Higher dimensions** — Add `max_depth` (Integer, 5–50) and `min_samples_split` (Integer, 2–20). How do the methods scale with 6 dimensions?
3. **Optuna comparison** — Run Optuna alongside GP-based optimization. Compare convergence curves.
4. **Acquisition function sweep** — Try `acq_func='LCB'` and `acq_func='PI'` in `gp_minimize`. How does the exploration–exploitation balance change?
5. **Simulated expensive evaluations** — Add `time.sleep(2)` inside `evaluate_params`. Now the wall-clock difference between methods becomes tangible.